# MicroLM — Pretraining on Kaggle T4 GPU

This notebook trains a **125M-parameter language model from scratch** on WikiText-103.

**Hardware:** Kaggle T4 (16GB VRAM) — free tier  
**Estimated time:** ~8 hours for 50,000 steps  
**Expected final loss:** ~2.8–3.2 (perplexity ~16–25)

## Steps
1. Clone repo + install deps
2. Download WikiText-103 and train BPE tokenizer
3. Preprocess text → binary token shards
4. Run pretraining
5. Download checkpoint

## 1. Setup

In [ ]:
# Verify GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# Clone the repo
!git clone https://github.com/lumixed/MicroLM.git
%cd MicroLM

In [ ]:
# Install dependencies
!pip install -r requirements.txt -q
print('Dependencies installed!')

## 2. Download Data & Train Tokenizer

In [ ]:
import os
os.makedirs('data/raw', exist_ok=True)

from datasets import load_dataset

print('Downloading WikiText-103...')
ds_train = load_dataset('wikitext', 'wikitext-103-raw-v1', split='train')
ds_val   = load_dataset('wikitext', 'wikitext-103-raw-v1', split='validation')

with open('data/raw/train.txt', 'w') as f:
    f.write('\n'.join(ds_train['text']))
with open('data/raw/val.txt', 'w') as f:
    f.write('\n'.join(ds_val['text']))

import os
train_mb = os.path.getsize('data/raw/train.txt') / 1e6
print(f'Train: {train_mb:.0f} MB')

In [ ]:
# Train BPE tokenizer (vocab_size=32000, takes ~2 min)
!python tokenizer/train_tokenizer.py \
    --input data/raw/ \
    --vocab-size 32000 \
    --output tokenizer/tokenizer.json
print('Tokenizer trained!')

## 3. Preprocess Text → Token Shards

In [ ]:
!python data/preprocess.py \
    --input data/raw/ \
    --tokenizer tokenizer/tokenizer.json \
    --output data/tokens/
print('Preprocessing done!')

In [ ]:
# Verify shards
import glob
shards = sorted(glob.glob('data/tokens/*.bin'))
print(f'Shards: {len(shards)}')
for s in shards[:5]:
    size_mb = os.path.getsize(s) / 1e6
    print(f'  {s}: {size_mb:.1f} MB')

## 4. Pretraining

In [ ]:
# (Optional) Log in to WandB for live training curves
# import wandb
# wandb.login()

In [ ]:
# Launch pretraining on T4!
# Expected: loss ~5.0 at step 0, ~2.8-3.2 at step 50,000
# Runtime: ~8-10 hours for full 50k steps
!python training/train.py --config training/configs/125m_config.yaml 2>&1 | tee training_log.txt

## 5. Evaluate & Download Checkpoint

In [ ]:
# List checkpoints
import glob
checkpoints = sorted(glob.glob('checkpoints/*.pt'))
for ckpt in checkpoints:
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f'{ckpt}: {size_mb:.0f} MB')

In [ ]:
# Quick perplexity check on validation set
!python eval/perplexity.py \
    --checkpoint checkpoints/microlm-125m-code-v1_final.pt \
    --tokenizer tokenizer/tokenizer.json \
    --text data/raw/val.txt \
    --max-tokens 100000

In [ ]:
# Quick generation test
!python inference/generate.py \
    --checkpoint checkpoints/microlm-125m-code-v1_final.pt \
    --tokenizer tokenizer/tokenizer.json \
    --prompt 'The history of artificial intelligence' \
    --max-new-tokens 100 \
    --temperature 0.8

In [ ]:
# Compress and download the best checkpoint + tokenizer
!tar -czf microlm_checkpoint.tar.gz \
    checkpoints/microlm-125m-code-v1_best.pt \
    tokenizer/tokenizer.json
print('Archive ready! Download microlm_checkpoint.tar.gz from Kaggle Output.')